In [1]:
import pyspark

In [2]:
#sc = pyspark.SparkContext.getOrCreate()

In [3]:
import os, sys
from pyspark.sql import SparkSession

# Java
os.environ["JAVA_HOME"] = "/opt/conda/envs/python312/lib/jvm"

# Force Python 3.12 everywhere
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["PYSPARK_PYTHON"] = sys.executable
# unable to use pyspark.SparkContext.getOrCreate() on jupyter.rwth-aachen.de
# so use jupyter local spark instead
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("ProcessMining")
    .config("spark.pyspark.driver.python", sys.executable)
    .config("spark.pyspark.python", sys.executable)
    .getOrCreate()
)
sc = spark.sparkContext

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/21 10:57:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


# Example (Word Count)

In [4]:
rdd = sc.textFile("data/example.txt") # Load the text file into pyspark

word_counts_rdd = (
    rdd
        .flatMap(lambda line: line.split(" ")) # Map a single value (line) to multiple values (words)
        .map(lambda word: (word, 1))
        .reduceByKey(lambda a,b: a + b) # Group all K,V pairs by key and sum up the counts
)

# Spark is lazy. The transformations are only computed as soon as we call `collect`
print(sorted(word_counts_rdd.collect()))

[('', 23), ('At', 12), ('Consetetur', 1), ('Duis', 6), ('Lorem', 26), ('Nam', 2), ('Stet', 12), ('Ut', 4), ('accumsan', 4), ('accusam', 12), ('ad', 4), ('adipiscing', 4), ('aliquam', 4), ('aliquip', 4), ('aliquyam', 12), ('amet,', 15), ('amet.', 11), ('assum.', 2), ('at', 4), ('augue', 4), ('autem', 6), ('blandit', 4), ('clita', 12), ('commodo', 4), ('congue', 2), ('consectetuer', 4), ('consequat,', 5), ('consequat.', 4), ('consetetur', 11), ('cum', 2), ('delenit', 4), ('diam', 27), ('dignissim', 4), ('dolor', 32), ('dolore', 25), ('dolores', 12), ('doming', 2), ('duis', 4), ('duo', 12), ('ea', 16), ('eirmod', 12), ('eleifend', 2), ('elit,', 4), ('elitr,', 12), ('enim', 4), ('eos', 12), ('erat', 4), ('erat,', 11), ('erat.', 1), ('eros', 4), ('esse', 5), ('est', 11), ('et', 56), ('eu', 5), ('euismod', 4), ('eum', 6), ('ex', 4), ('exerci', 4), ('facer', 2), ('facilisi.', 4), ('facilisis', 4), ('facilisis.', 1), ('feugait', 4), ('feugiat', 5), ('gubergren,', 12), ('hendrerit', 5), ('id', 

# DFG Discovery

## a)

In [5]:
from pm4py import view_dfg, save_vis_dfg

map_1 ∈ (N_0 × S) → (C x (T x A)), 𝒓educe_1 ∈ (C x (T x A)) → (C X A*) 
map_2 ∈ (C X A*) → ((A x A) x N_0), 𝒓educe_2 ∈ ((A x A) x N_0) → ((A x A) x N_0)

The computation of the Directly-Follows Graph (DFG) is divided into two MapReduce jobs.

In the first job, the map function groups events by their case id. It takes the line number and the whole log line as input, extracts the case id, activity, and timestamp, and outputs the activity and timestamp group by the case id.
The reduce function receives all activities and timestamps belonging to the same case, sorts them according to the timestamps, and outputs the ordered sequence of activities for each case.

In the second job, the map function processes the sorted activity sequence of each case and emits all directly-follows activity pairs. Each pair is emitted with a count of 1.
The reduce function groups identical activity pairs and sums their counts, producing the final DFG where each edge represents a directly-follows relation and its frequency.

### (ii)

In [6]:
def map_first_job(line):
    case_id,activity,timestamp,other = line.split(',')
    return case_id,(float(timestamp),activity)

def reduce_first_job(events):
    sorted_activities = []
    for event in sorted(events):
        timestamp, activity = event
        sorted_activities.append(activity)
    return sorted_activities

def map_second_job(events):
    activities_pair = []
    case_id, sorted_activities = events
    previous_activity = ""
    for activity in sorted_activities:
        if(previous_activity == ""):
            previous_activity = activity
        else:
            activities_pair.append(((previous_activity,activity),1))
            previous_activity = activity
    return activities_pair


In [7]:

rdd = sc.textFile("data/event_log.csv.gz")
DFG = (
    rdd
        .map(lambda line: map_first_job(line)) 
        .groupByKey()
        .mapValues(reduce_first_job)
        .flatMap(lambda event: map_second_job(event))
        .reduceByKey(lambda x, y: x + y)
)
DFG.takeOrdered(10, key=lambda x: -x[1])

[(('Create Fine', 'Send Fine'), 100499),
 (('Send Fine', 'Insert Fine Notification'), 76864),
 (('Insert Fine Notification', 'Add penalty'), 69602),
 (('Add penalty', 'Send for Credit Collection'), 57182),
 (('Create Fine', 'Payment'), 46952),
 (('Add penalty', 'Payment'), 18621),
 (('Payment', 'Payment'), 4306),
 (('Payment', 'Add penalty'), 3902),
 (('Insert Fine Notification', 'Payment'), 3886),
 (('Send Fine', 'Payment'), 3305)]

In [8]:
from graphviz import Digraph

edges = DFG.collect()

dot = Digraph("DirectlyFollowsGraph", format="png")
dot.attr(rankdir="LR")
dot.attr("node", shape="box")

for edge in edges:
    (activity_x, activity_y), count = edge
    dot.edge(str(activity_x), str(activity_y), label=str(count))

dot.render("output/dfg")

'output/dfg.png'

First, we parse each log line and extract case id, activity, and timestamp, emitting (case_id, (timestamp, activity)). Next, groupByKey groups all events belonging to the same case. Then, mapValues sorts the events of each case by timestamp to obtain the ordered sequence of activities. After that, flatMap extracts all directly-follows relations by emitting each adjacent activity pair from the sequence with count 1. Finally, reduceByKey sums these counts per activity pair to count the occurrences of the directly-follows relations.

## (iii)

In [9]:

DFG_filtered = (
    rdd
        .map(lambda line: map_first_job(line)) 
        .groupByKey()
        .mapValues(reduce_first_job)
        .flatMap(lambda event: map_second_job(event))
        .reduceByKey(lambda x, y: x + y)
        .filter(lambda x: x[1] >= 100)
)
from graphviz import Digraph

edges = DFG_filtered.collect()

dot = Digraph("DirectlyFollowsGraph", format="png")
dot.attr(rankdir="LR")
dot.attr("node", shape="box")

for edge in edges:
    (activity_x, activity_y), count = edge
    dot.edge(str(activity_x), str(activity_y), label=str(count))

dot.render("output/dfg_filtered")

'output/dfg_filtered.png'

Adding a filter transformation removes directly-follows relations with a frequency smaller than 100. By applying filter(lambda x: x[1] >= 100), all arcs whose frequency is below 100 are filtered out, resulting in a simplified Directly-Follows Graph.

# (iv)

In [10]:
def map_job(line):
    case_id,activity,timestamp,resource = line.split(',')
    return activity,resource

def reduce_job(event):
    resource_dict = {}
    activity,resource_list = event
    for resource in resource_list:
        if resource not in resource_dict:
            resource_dict[resource] = 0
        resource_dict[resource] += 1

    max_resource, max_count = max(resource_dict.items(), key=lambda x: x[1])
        
    return activity,(max_resource, max_count)

DFG_resource = (
    rdd
        .map(lambda line: map_job(line)) 
        .groupByKey()
        .map(lambda event : reduce_job(event))
)

DFG_resource.collect()


[('Create Fine', ('R-538', 8608)),
 ('Send Fine', ('MISSING', 103987)),
 ('Insert Fine Notification', ('MISSING', 79860)),
 ('Add penalty', ('MISSING', 79860)),
 ('Send for Credit Collection', ('MISSING', 59013)),
 ('Payment', ('MISSING', 77601)),
 ('Insert Date Appeal to Prefecture', ('MISSING', 4188)),
 ('Send Appeal to Prefecture', ('MISSING', 4141)),
 ('Receive Result Appeal from Prefecture', ('MISSING', 999)),
 ('Notify Result Appeal to Offender', ('MISSING', 896)),
 ('Appeal to Judge', ('R-0', 555))]

First, each log line is parsed to extract the activity and resource, emitting pairs of the form 
(activity,resource)
(activity,resource). Next, groupByKey groups all events belonging to the same activity. Then, the reduce step counts how many times each resource executes the activity. Finally, for each activity, the resource that most frequently executes it is selected and emitted.